# SCADA alarm grounded root-cause demo

Ingest the knowledge base, run the five retrieval sanity queries, then pass a simulated alarm through `RootCauseAgent.propose`.

Requires Ollama with `qwen2.5:3b` (`ollama pull qwen2.5:3b`). Run from the repository root with the project venv.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "configs" / "rag.yaml").exists():
    raise SystemExit("Run this notebook from the repository root.")
sys.path.insert(0, str(ROOT / "src"))

from scada_rag.config import RagConfig
from scada_rag.ingest import ingest
from scada_rag.retrieve import retrieve
from scada_rag.sanity import SANITY_QUERIES

config = RagConfig.load()
count = ingest(config)
print(f"{count} chunks in {config.persist_dir}")

In [ ]:
for query, expected in SANITY_QUERIES:
    hits = retrieve(query, config, top_k=3)
    sources = [hit.source for hit in hits]
    flag = "PASS" if expected in sources else "FAIL"
    print(f"{flag}  {query}")
    print(f"     expected {expected}")
    for hit in hits:
        print(f"     {hit.score:.3f}  {hit.source}  /  {hit.section}")

In [ ]:
from scada_rag.agent import RootCauseAgent, format_answer
from scada_rag.alarms import AlarmPayload, ShapSensor
from scada_rag.eval_alarms import load_scenarios

scenarios = load_scenarios(ROOT / "eval" / "alarms.yaml")
demo = next(s for s in scenarios if s.id == "hss_bearing_partial_load")

agent = RootCauseAgent(config)
answer = agent.propose(demo.payload)
print(format_answer(answer))